# Spatial Lag and Spatial Error Models

We begin where most regression analyses begin, with ordinary least squares, and ask whether its residuals carry spatial structure. When they do, the Bayesian Lagrange multiplier tests tell us whether the dependence belongs in the outcome, which calls for a spatial lag model, or in the errors, which calls for a spatial error model. The example and its interpretation, premature mortality in Chicago tracts and the immigrant paradox, follow Luc Anselin's course material.

In [ ]:
import arviz as az
import geopandas as gpd
import pandas as pd
from esda import Moran
from libpysal.graph import Graph
from neighbayes.models import OLS, SAR, SEM

az.rcParams["stats.ci_prob"] = 0.95

In [ ]:
chi = gpd.read_file("data/Chicago-SDOH/Chi-SDOH.shp")
chi["ypll"] = chi["YPLL_rate"] / 1000  # premature mortality, in thousands of years of life lost
graph = Graph.build_contiguity(chi, rook=False).transform("r")

In [ ]:
ax = chi.plot(
    "ypll", scheme="quantiles", k=5, cmap="OrRd", legend=True,
    legend_kwds={"loc": "lower left"}, figsize=(7, 7),
)
ax.set_axis_off()

Premature mortality is far from randomly distributed: the highest rates cluster on the South and West Sides. The question for a regression is whether that clustering is explained by the covariates or survives them.

## OLS

We start with the percent Black and percent Hispanic population alone.

In [ ]:
fit_kwargs = dict(draws=1000, tune=1000, chains=4, random_seed=42)

ols_race = OLS(formula="ypll ~ Blk14P + Hisp14P", data=chi, W=graph)
ols_race.fit(**fit_kwargs)
ols_race.summary(var_names=["beta"], round_to=3)

Both coefficients are positive, and both intervals lie well away from zero. Adding the economic hardship index changes the picture.

In [ ]:
formula = "ypll ~ Blk14P + Hisp14P + HIS_ct"

ols = OLS(formula=formula, data=chi, W=graph)
ols.fit(**fit_kwargs)
ols.summary(var_names=["beta"], round_to=3)

Conditional on economic hardship, the coefficient on the Hispanic share turns negative. Tracts with larger Hispanic populations have lower premature mortality than their hardship would predict, which is the immigrant paradox. The positive coefficient in the first model reflected the hardship that Hispanic tracts share, not the Hispanic share itself.

## Spatial structure in the residuals

The morning's tools answer the first spatial question: are the residuals spatially autocorrelated?

In [ ]:
ax = chi.assign(residual=ols.residuals()).plot(
    "residual", scheme="quantiles", k=5, cmap="RdBu_r", legend=True, figsize=(7, 7)
)
ax.set_axis_off()

In [ ]:
for name, model in [("race and ethnicity", ols_race), ("with hardship", ols)]:
    mi = Moran(model.residuals(), graph)
    print(f"{name}: I = {mi.I:.3f}, pseudo p = {mi.p_sim:.3f}")

Hardship accounts for much of the clustering, and Moran's I falls by more than half when it enters the model. What remains is still more than chance would produce. Moran's I tells us the residuals are dependent, but not why, and the two usual explanations call for different models.

## Bayesian Lagrange multiplier tests

A Lagrange multiplier test asks whether a model would fit better if a parameter it fixes at zero were set free: $\rho$ for a spatial lag, $\lambda$ for a spatial error. The classical tests evaluate that question once, at the OLS estimates. The Bayesian versions evaluate it at every posterior draw, which yields a posterior distribution for each test statistic.

In [ ]:
ols.spatial_diagnostics()

Each row is a test, and each column summarizes the posterior distribution of its statistic: the mean, the median, and the 95% interval. The p value compares the posterior mean with the $\chi^2$ reference distribution.

- `LM-Lag` and `LM-Error` test for an omitted lag and an omitted spatial error process.
- `Robust-LM-Lag` and `Robust-LM-Error` test for each while allowing for the other. When both simple tests reject, the robust pair is meant to separate them.
- `LM-SDM-Joint` and `LM-SLX-Error-Joint` test jointly for the spatial process and for lagged covariates, the terms of the Durbin models in the next notebook.

Both simple tests reject, and neither robust test does, so the robust pair cannot tell the lag from the error here. The decision tree resolves the tie with the smaller of the two simple p values.

In [ ]:
ols.spatial_diagnostics_decision()

The tree selects the spatial lag model, but by a narrow route. The same tests on the model without hardship tell a different story.

In [ ]:
ols_race.spatial_diagnostics_decision(format="model")

Without hardship, only the robust error test rejects, and the tree selects the spatial error model. A covariate that is itself clustered in space, left out of the model, leaves exactly this signature: dependence in the errors. The diagnostics are a statement about a specification, not about the data alone.

## The spatial lag model

$$y = \rho W y + X\beta + \varepsilon$$

In [ ]:
sar = SAR(formula=formula, data=chi, W=graph)
sar.fit(**fit_kwargs)
sar.summary(round_to=3)

$\rho$ is positive and its interval excludes zero. Its presence changes what the coefficients mean. A change in hardship in one tract raises mortality there, which raises it in the neighboring tracts through $\rho W y$, which feeds back to the first tract, and so on, so $\beta$ is no longer the effect of a covariate on the outcome. The next notebook computes the effects themselves.

The lag absorbs the dependence that OLS left behind.

In [ ]:
mi = Moran(sar.residuals(), graph)
print(f"I = {mi.I:.3f}, pseudo p = {mi.p_sim:.3f}")

The tests on a fitted lag model ask what it might still be missing: a spatial error process, or the neighbors' covariates.

In [ ]:
sar.spatial_diagnostics()

The simple test for a remaining error process does not reject, while the test for lagged covariates, `LM-WX`, does. The decision tree reads the same tests and moves to the spatial Durbin model.

In [ ]:
sar.spatial_diagnostics_decision(format="model")

## The spatial error model

$$y = X\beta + u, \qquad u = \lambda W u + \varepsilon$$

The error model treats spatial dependence as a nuisance: unobserved factors that cluster in space. The coefficients keep their OLS meaning, and $\lambda$, reported as `lam`, measures the clustering of the unobserved factors.

In [ ]:
sem = SEM(formula=formula, data=chi, W=graph)
sem.fit(**fit_kwargs)
sem.summary(round_to=3)

In [ ]:
sem.spatial_diagnostics()

The simple tests mirror the lag model's: they detect no omitted lag but do detect lagged covariates, and the tree moves to the spatial Durbin error model.

In [ ]:
sem.spatial_diagnostics_decision(format="model")

In [ ]:
pd.DataFrame(
    {name: model.summary()["mean"] for name, model in [("OLS", ols), ("SAR", sar), ("SEM", sem)]}
).loc[["Blk14P", "Hisp14P", "HIS_ct"]].round(3)

The three models agree on the signs and roughly on the magnitudes, including the negative coefficient on the Hispanic share. They disagree about where the spatial dependence comes from, and both spatial models' diagnostics point the same way: the neighbors' covariates belong in the model. That is the Durbin specification, which we take up after the break.

<div class="alert alert-success" style="font-size:120%">
<b>Exercise</b>: <br>
Build a graph from the six nearest neighbors of each tract, <code>Graph.build_knn(chi.representative_point(), k=6)</code>, row-standardize it, and rerun the OLS diagnostics. Does the decision tree still choose the spatial lag model?
</div>

In [ ]:
# %load solutions/07.py